In [1]:
import pandas as pd
import yt_dlp
import os
import time

# 윈도우 경로 인식 오류를 막기 위해 raw string(r) 사용 또는 슬래시(/) 사용
DOWNLOAD_DIR = "data"

# 이미 만드셨다고 했지만, 안전을 위해 체크 로직은 남겨둡니다.
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print(f"✅ 저장 경로 세팅 완료: {DOWNLOAD_DIR}")

✅ 저장 경로 세팅 완료: data


In [2]:
def download_shorts_batch(video_url, output_dir):
    """
    단일 URL을 받아 영상을 다운로드하는 함수입니다.
    """
    ydl_opts = {
        # 1. 해상도 및 포맷 제한: 1080p 이하의 mp4 포맷만 강제 (연산 속도 및 용량 최적화)
        'format': 'bestvideo[ext=mp4][height<=1080]+bestaudio[ext=m4a]/best[ext=mp4][height<=1080]',
        
        # 2. 파일명 규칙: 기존 CSV의 PK와 일치시키기 위해 video_id.mp4 로 저장
        'outtmpl': f'{output_dir}/%(id)s.%(ext)s',
        
        # 3. 에러 무시: 삭제되거나 비공개 처리된 영상이 있어도 파이프라인이 멈추지 않음 (매우 중요)
        'ignoreerrors': True,
        
        # 터미널 출력 간소화 (진행률만 표시)
        'quiet': True,
        'no_warnings': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # 다운로드 실행 (메타데이터를 먼저 뽑아 실제 다운로드된 파일명 확인)
            info = ydl.extract_info(video_url, download=True)
            
            # info가 None이면 비공개/삭제된 영상
            if info is None:
                return False
            
            return True
            
    except Exception as e:
        print(f"❌ 다운로드 중단 에러: {e}")
        return False

In [ ]:
# 1. 형욱 님이 2개의 샘플을 넣어두신 CSV 파일 경로를 입력하세요.
csv_file_path = "shorts-analysis_test-1.csv" # 실제 파일명으로 변경해주세요!

# 2. CSV 파일 읽기
try:
    df = pd.read_csv(csv_file_path)
    print(f"📊 총 {len(df)}개의 샘플 영상 리스트를 불러왔습니다.")
except FileNotFoundError:
    print("❌ CSV 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    df = pd.DataFrame()

# 3. 다운로드 루프 실행
success_count = 0

for index, row in df.iterrows():
    vid_id = row['video_id']
    vid_url = row['final_url'] # 알려주신 컬럼명 반영
    
    print(f"[{index+1}/{len(df)}] 다운로드 시도: {vid_id}")
    
    # 중복 체크
    if os.path.exists(f"{DOWNLOAD_DIR}/{vid_id}.mp4"):
        print(f" ⏩ 이미 존재하는 파일입니다: {vid_id}")
        success_count += 1
        continue
        
    # 다운로드 실행
    is_success = download_shorts_batch(vid_url, DOWNLOAD_DIR)
    
    if is_success:
        print(f" ✅ 성공: {vid_id}.mp4")
        success_count += 1
    else:
        print(f" ⚠️ 실패: {vid_id}")
        
    time.sleep(1)

print(f"\n🎉 샘플 다운로드 완료! (성공: {success_count}개)")

📊 총 6개의 샘플 영상 리스트를 불러왔습니다.
[1/2] 다운로드 시도: knCQBlBKSRM
 ✅ 성공: knCQBlBKSRM.mp4                                   
[2/2] 다운로드 시도: NCVxpXxTMSU
 ✅ 성공: NCVxpXxTMSU.mp4                                   
[3/2] 다운로드 시도: LRmLOsmMHts
 ✅ 성공: LRmLOsmMHts.mp4                                     
[4/2] 다운로드 시도: twi9zUxsXu0
 ✅ 성공: twi9zUxsXu0.mp4                                     
[5/2] 다운로드 시도: Xfu1kCID0Ls
 ✅ 성공: Xfu1kCID0Ls.mp4                                   
[6/2] 다운로드 시도: MIJR3Dm0YOk
 ✅ 성공: MIJR3Dm0YOk.mp4                                     

🎉 샘플 다운로드 완료! (성공: 6개)
